In [1]:
import json
import math
from pathlib import Path

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split

# Locate and load the cleaned dataset
ml_data_path = None

for ml_candidate in [Path.cwd(), *Path.cwd().parents]:
    ml_possible_path = ml_candidate / "data" / "bugradar_issues_cleaned.json"
    if ml_possible_path.is_file():
        ml_data_path = ml_possible_path
        break

if ml_data_path is None:
    raise FileNotFoundError(
        "Could not find data/bugradar_issues_cleaned.json. "
        "Run this notebook from inside the BugRadar project."
    )

with ml_data_path.open("r", encoding="utf-8") as ml_file:
    ml_source_data = json.load(ml_file)

if not isinstance(ml_source_data, list) or not all(
    isinstance(doc, dict) for doc in ml_source_data
):
    raise ValueError("The cleaned JSON must contain a list of issue documents.")

ml_required_fields = {"title", "body", "issue_category"}
ml_missing_fields = ml_required_fields - set().union(
    *(doc.keys() for doc in ml_source_data)
) if ml_source_data else ml_required_fields

if ml_missing_fields:
    raise ValueError(f"Dataset is missing required fields: {ml_missing_fields}")

# Combine title and description, and prepare category labels
ml_classification_frame = pd.DataFrame([
    {
        "text": " ".join(
            value.strip()
            for value in (doc.get("title"), doc.get("body"))
            if isinstance(value, str) and value.strip()
        ),
        "category": doc.get("issue_category")
    }
    for doc in ml_source_data
])

ml_classification_frame = ml_classification_frame[
    ml_classification_frame["category"].apply(
        lambda value: isinstance(value, str) and bool(value.strip())
    )
].copy()

ml_classification_text = ml_classification_frame["text"].tolist()
ml_classification_target = ml_classification_frame["category"].tolist()

# Exclude issues with no usable title or description
ml_nonempty_text = [
    bool(text.strip()) for text in ml_classification_text
]

ml_classification_text = [
    text for text, keep in zip(ml_classification_text, ml_nonempty_text)
    if keep
]
ml_classification_target = [
    target for target, keep in zip(ml_classification_target, ml_nonempty_text)
    if keep
]

ml_classification_count = len(ml_classification_target)

if ml_classification_count < 2:
    raise ValueError("Not enough valid issue records for classification.")

ml_class_counts = pd.Series(ml_classification_target).value_counts()

if len(ml_class_counts) < 2:
    raise ValueError("Classification requires at least two issue categories.")

ml_test_count = math.ceil(0.2 * ml_classification_count)

# Stratify when every category has enough records for both splits
ml_can_stratify = (
    ml_class_counts.min() >= 2
    and ml_test_count >= len(ml_class_counts)
    and (ml_classification_count - ml_test_count) >= len(ml_class_counts)
)

ml_train_text, ml_test_text, ml_train_target, ml_test_target = train_test_split(
    ml_classification_text,
    ml_classification_target,
    test_size=0.2,
    random_state=42,
    stratify=ml_classification_target if ml_can_stratify else None
)

if len(set(ml_train_target)) < 2:
    raise ValueError(
        "The training split contains fewer than two categories. "
        "Add more records or adjust the split."
    )

# Convert text into TF-IDF features and train Logistic Regression
ml_classification_vectorizer = TfidfVectorizer()
ml_train_features = ml_classification_vectorizer.fit_transform(ml_train_text)
ml_test_features = ml_classification_vectorizer.transform(ml_test_text)

ml_classification_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)
ml_classification_model.fit(ml_train_features, ml_train_target)

ml_classification_predictions = ml_classification_model.predict(
    ml_test_features
)

# Evaluate the classification model
print("TASK 5: ISSUE CLASSIFICATION")
print(f"Valid records: {ml_classification_count}")
print(f"Training records: {len(ml_train_target)}")
print(f"Testing records: {len(ml_test_target)}")
print(f"Accuracy:  {accuracy_score(ml_test_target, ml_classification_predictions):.4f}")
print(
    f"Precision: {precision_score(ml_test_target, ml_classification_predictions, average='weighted', zero_division=0):.4f}"
)
print(
    f"Recall:    {recall_score(ml_test_target, ml_classification_predictions, average='weighted', zero_division=0):.4f}"
)
print(
    f"F1-score:  {f1_score(ml_test_target, ml_classification_predictions, average='weighted', zero_division=0):.4f}"
)

TASK 5: ISSUE CLASSIFICATION
Valid records: 5000
Training records: 4000
Testing records: 1000
Accuracy:  1.0000
Precision: 1.0000
Recall:    1.0000
F1-score:  1.0000


In [3]:
import math
from datetime import datetime, timezone
from numbers import Real

from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.model_selection import train_test_split

if "ml_source_data" not in globals():
    raise RuntimeError("Run the Task 5 cell first to load the dataset.")

def parse_ml_timestamp(value):
    if not isinstance(value, str) or not value.strip():
        return None

    try:
        timestamp = datetime.fromisoformat(
            value.strip().replace("Z", "+00:00")
        )
        if timestamp.tzinfo is None:
            timestamp = timestamp.replace(tzinfo=timezone.utc)
        return timestamp
    except ValueError:
        return None


ml_regression_text = []
ml_regression_target = []

for doc in ml_source_data:
    # Use only closed issues with a valid, nonnegative duration
    if not isinstance(doc.get("state"), str) or doc["state"].casefold() != "closed":
        continue

    duration = doc.get("resolution_hours")

    if (
        not isinstance(duration, Real)
        or isinstance(duration, bool)
        or not math.isfinite(float(duration))
        or duration < 0
    ):
        continue

    created_at = parse_ml_timestamp(doc.get("created_at"))
    closed_at = parse_ml_timestamp(doc.get("closed_at"))

    if created_at is None or closed_at is None or closed_at < created_at:
        continue

    # Check that the stored duration agrees with the timestamps.
    # The cleaned dataset stores resolution_hours rounded to 2 decimals.
    calculated_hours = (closed_at - created_at).total_seconds() / 3600

    if not math.isclose(
        float(duration), calculated_hours, abs_tol=0.011, rel_tol=0
    ):
        continue

    text = " ".join(
        value.strip()
        for value in (doc.get("title"), doc.get("body"))
        if isinstance(value, str) and value.strip()
    )

    if not text:
        continue

    ml_regression_text.append(text)
    ml_regression_target.append(float(duration))

if len(ml_regression_target) < 6:
    raise ValueError(
        "Not enough valid closed issues for regression. "
        "At least 6 records with usable text and valid timestamps are required."
    )

# Split data before fitting TF-IDF to avoid data leakage
ml_reg_train_text, ml_reg_test_text, ml_reg_train_target, ml_reg_test_target = (
    train_test_split(
        ml_regression_text,
        ml_regression_target,
        test_size=0.2,
        random_state=42
    )
)

ml_regression_vectorizer = TfidfVectorizer()
ml_reg_train_features = ml_regression_vectorizer.fit_transform(
    ml_reg_train_text
)
ml_reg_test_features = ml_regression_vectorizer.transform(
    ml_reg_test_text
)

# Train the Random Forest regression model
ml_regression_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
ml_regression_model.fit(ml_reg_train_features, ml_reg_train_target)

ml_regression_predictions = ml_regression_model.predict(
    ml_reg_test_features
)

# Evaluate the regression model
print("\nTASK 6: RESOLUTION-TIME PREDICTION")
print(f"Valid closed issues: {len(ml_regression_target)}")
print(f"Training records: {len(ml_reg_train_target)}")
print(f"Testing records: {len(ml_reg_test_target)}")
print(
    f"Mean Absolute Error (hours): "
    f"{mean_absolute_error(ml_reg_test_target, ml_regression_predictions):.2f}"
)


TASK 6: RESOLUTION-TIME PREDICTION
Valid closed issues: 3857
Training records: 3085
Testing records: 772
Mean Absolute Error (hours): 176.76
